# Rayleigh Similarity Problem

This notebook compares MI-DL, SI-DL, and the known Rayleigh similarity coordinate. It regenerates the synthetic velocity-profile data, reruns both searches, and retains the raw-coordinate comparison rather than a fitted GPR figure.


## 1. Imports and settings

Configure the generated dataset, dimensional matrix, repeated searches, and output folders. CSV results are written to `csv/`; only `raw.png` and `summary.png` are retained in `figures/`.


In [ ]:
from __future__ import annotations

import os

import sys

import warnings

from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-codex-cache")

os.environ.setdefault("MPLBACKEND", "Agg")

warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.optimize import differential_evolution

from scipy.special import erf

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

from sklearn.metrics import mean_squared_error

from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = Path.cwd()

ROOT = OUTPUT_DIR.parent

for module_dir in [ROOT / "Compare" / "MI-DL", ROOT / "SI-DL-main"]:
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import midl

import SI_DL

RANDOM_STATE = 42

FIG_DIR = OUTPUT_DIR / "figures"

CSV_DIR = OUTPUT_DIR / "csv"

INPUT_COLUMNS = ["U", "y", "t", "nu"]

VARIABLE_LABELS = ["U", "y", "t", "nu"]

MI_K_NEIGHBORS = 6

MI_DE_MAXITER = 260

MI_RESTART_SEEDS = [0, 42, 101]

SI_BANDWIDTH = 0.02

SI_MAXITER = 70

SI_POPSIZE = 8

TEST_SIZE = 0.25

D_IN = np.array([[1, 1, 0, 2], [-1, 0, 1, -1]], dtype=float)

PHYSICAL_BASIS = np.array(
    [
        [1.0, -1.0, 1.0, 0.0],
        [0.0, 1.0, -0.5, -0.5],
    ],
    dtype=float,
)

KNOWN_RAYLEIGH = np.array([0.0, 1.0, -0.5, -0.5], dtype=float)


## 2. Generate Rayleigh data and coordinate utilities

Construct the velocity-profile dataset, normalize candidate exponents, and evaluate information metrics.


In [ ]:
def velocity_profile_rayleigh(y: np.ndarray, nu: np.ndarray, U: np.ndarray, t: np.ndarray) -> np.ndarray:
    return U * (1.0 - erf(y / (2.0 * np.sqrt(nu * t))))

def generate_rayleigh_data(
    n_velocity_viscosity_samples: int = 30,
    n_y: int = 20,
    n_t: int = 15,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    y_values = np.linspace(1e-3, 1.0, n_y)
    t_values = np.linspace(0.01, 5.0, n_t)
    U_samples = rng.uniform(0.5, 1.0, n_velocity_viscosity_samples)
    nu_samples = rng.uniform(1e-3, 1e-2, n_velocity_viscosity_samples)

    rows = []
    for U_current, nu_current in zip(U_samples, nu_samples):
        yy, tt = np.meshgrid(y_values, t_values, indexing="ij")
        U = np.full(yy.size, U_current)
        nu = np.full(yy.size, nu_current)
        u = velocity_profile_rayleigh(yy.ravel(), nu, U, tt.ravel())
        rows.append(
            pd.DataFrame(
                {
                    "U": U,
                    "y": yy.ravel(),
                    "t": tt.ravel(),
                    "nu": nu,
                    "u": u,
                    "u_over_U": u / U,
                }
            )
        )
    return pd.concat(rows, ignore_index=True)

def normalize_exponents(exponents: np.ndarray) -> np.ndarray:
    row = np.asarray(exponents, dtype=float).reshape(-1)
    scale = float(np.max(np.abs(row)))
    if scale <= 1e-12:
        return row
    out = row / scale
    first = np.flatnonzero(np.abs(out) > 1e-10)
    if first.size and out[first[0]] < 0.0:
        out = -out
    return out

def log_pi_from_exponents(X: np.ndarray, exponents: np.ndarray) -> np.ndarray:
    return np.log(np.asarray(X, dtype=float)) @ np.asarray(exponents, dtype=float).reshape(-1)

def formula_from_exponents(exponents: np.ndarray, decimals: int = 4) -> str:
    terms = []
    for label, value in zip(VARIABLE_LABELS, np.asarray(exponents).reshape(-1)):
        value = float(value)
        if abs(value) < 5e-5:
            continue
        terms.append(f"{label}^{value:.{decimals}f}")
    return " * ".join(terms)

def information_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    info = midl.information_lower_bound(
        np.asarray(feature, dtype=float).reshape(-1, 1),
        Y,
        k_neighbors=MI_K_NEIGHBORS,
        random_state=RANDOM_STATE,
    )
    return {
        "mutual_information": float(info["mutual_information"]),
        "epsilon_lb_normalized": float(info["epsilon_lb"] / np.var(Y, ddof=0)),
    }


## 3. MI-DL and SI-DL searches

Optimize the information and covariance objectives in the dimensional null space.


In [ ]:
def run_midl_search(X: np.ndarray, Y: np.ndarray, search_idx: np.ndarray) -> dict:
    basis, _ = midl.calc_basis(D_IN)
    log_pi_basis = np.log(X[search_idx]) @ basis
    Y_search = Y[search_idx]
    rows = []
    best = None
    for seed in MI_RESTART_SEEDS:
        model = midl.MIDL(
            k_neighbors=MI_K_NEIGHBORS,
            de_maxiter=MI_DE_MAXITER,
            random_state=seed,
        )
        w, _search_mi, _ = model._optimize_direction_in_subspace(
            log_pi_basis,
            Y_search,
            np.eye(basis.shape[1]),
        )
        exponents = normalize_exponents(np.asarray(basis @ w).reshape(-1))
        feature_search = log_pi_from_exponents(X[search_idx], exponents)
        mi_search = information_metrics(feature_search, Y_search)
        row = {
            "seed": seed,
            "mi_search": mi_search["mutual_information"],
            "epsilon_lb_search_normalized": mi_search["epsilon_lb_normalized"],
            "formula": formula_from_exponents(exponents),
            "exponents": exponents,
        }
        rows.append(row)
        if best is None or row["mi_search"] > best["mi_search"]:
            best = row
    audit = pd.DataFrame([{k: v for k, v in row.items() if k != "exponents"} for row in rows])
    audit = audit.sort_values("mi_search", ascending=False)
    return {
        "exponents": best["exponents"],
        "best_seed": int(best["seed"]),
        "search_score": float(best["mi_search"]),
        "restart_audit": audit,
    }

def params_to_exponents(params: np.ndarray) -> np.ndarray:
    return normalize_exponents(np.asarray(params, dtype=float).reshape(-1) @ PHYSICAL_BASIS)

def run_sidl_search(X: np.ndarray, Y: np.ndarray, search_idx: np.ndarray) -> dict:
    X_search = X[search_idx]
    Y_search = Y[search_idx]

    def objective(params: np.ndarray) -> float:
        try:
            exponents = params_to_exponents(params)
            feature = log_pi_from_exponents(X_search, exponents)
            score = SI_DL.explained_variance_score(feature, Y_search, bandwidth=SI_BANDWIDTH)["S_cov"]
        except Exception:
            return 1e6
        if not np.isfinite(score):
            return 1e6
        return -float(score)

    result = differential_evolution(
        objective,
        bounds=[(-2.0, 2.0)] * PHYSICAL_BASIS.shape[0],
        maxiter=SI_MAXITER,
        popsize=SI_POPSIZE,
        seed=RANDOM_STATE,
        polish=True,
        updating="immediate",
        workers=1,
    )
    exponents = params_to_exponents(result.x)
    score = SI_DL.explained_variance_score(
        log_pi_from_exponents(X_search, exponents),
        Y_search,
        bandwidth=SI_BANDWIDTH,
    )
    return {
        "optimizer_result": result,
        "exponents": exponents,
        "search_score": float(score["S_cov"]),
        "score": score,
    }

def score_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    sidl_score = SI_DL.explained_variance_score(feature, Y, bandwidth=SI_BANDWIDTH)
    mi_raw = information_metrics(feature, Y)
    return {
        **mi_raw,
        "S_cov": float(sidl_score["S_cov"]),
        "sidl_error": float(1.0 - sidl_score["S_cov"]),
        "sidl_bandwidth": float(sidl_score["bandwidth"]),
        "sidl_n_retained": int(sidl_score["n_retained"]),
    }


## 4. GPR metrics and summary table

GPR is still evaluated numerically for the summary CSV, but no GPR fit image is created.


In [ ]:
def gpr_fit(feature: np.ndarray, Y: np.ndarray, gpr_idx: np.ndarray) -> dict:
    feature = np.asarray(feature, dtype=float).reshape(-1, 1)
    Y = np.asarray(Y, dtype=float).reshape(-1)
    train_rel, test_rel = train_test_split(
        np.arange(gpr_idx.size),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    train_idx = gpr_idx[train_rel]
    test_idx = gpr_idx[test_rel]
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-3, 1e3))
    kernel += WhiteKernel(1.0, (1e-8, 1e3))
    model = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-8,
            normalize_y=True,
            optimizer=None,
            n_restarts_optimizer=0,
            random_state=RANDOM_STATE,
        ),
    )
    model.fit(feature[train_idx], Y[train_idx])
    y_pred = model.predict(feature[test_idx])
    mse_raw = float(mean_squared_error(Y[test_idx], y_pred))
    scale = float(np.var(Y[train_idx], ddof=0))
    return {
        "model": model,
        "train_idx": train_idx,
        "test_idx": test_idx,
        "y_pred_test": y_pred,
        "mse_raw": mse_raw,
        "mse_normalized": mse_raw / scale,
        "rmse_normalized": float(np.sqrt(mse_raw / scale)),
        "error_scale_var_y_train": scale,
    }

def plot_summary_table(summary: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(16.8, 5.6), dpi=220)
    ax.axis("off")
    fig.patch.set_facecolor("white")
    ax.text(0.5, 0.96, "Rayleigh comparison, k=6", ha="center", va="top", fontsize=18, weight="bold")
    rows = []
    for _, row in summary.iterrows():
        rows.append(
            [
                row["method"],
                row["formula"],
                f"{row['mutual_information']:.4f}",
                f"{row['epsilon_lb_normalized']:.5f}",
                f"{row['S_cov']:.4f}",
                f"{row['gpr_mse_normalized']:.5f}",
            ]
        )
    table = ax.table(
        cellText=rows,
        colLabels=["Method", "Found pi", "MI k=6", "epsilon_LB/Var", "S_cov", "GPR norm. MSE"],
        cellLoc="center",
        colLoc="center",
        colWidths=[0.13, 0.48, 0.09, 0.10, 0.09, 0.11],
        bbox=[0.02, 0.08, 0.96, 0.78],
    )
    table.auto_set_font_size(False)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("#3f3f46")
        cell.set_linewidth(0.85)
        cell.PAD = 0.03
        text = cell.get_text()
        if r == 0:
            cell.set_facecolor("#1f2937")
            text.set_color("white")
            text.set_weight("bold")
            text.set_fontsize(10.5)
        else:
            cell.set_facecolor("#f8fafc" if r % 2 == 1 else "#ffffff")
            text.set_fontsize(8.2 if c == 1 else 10.0)
            if c == 1:
                text.set_ha("left")
            if c == 0:
                text.set_weight("bold")
    fig.savefig(FIG_DIR / "summary.png", bbox_inches="tight", facecolor="white")
    plt.close(fig)


## 5. Run the comparison

Execute both searches, compare with the known Rayleigh coordinate, and save the compact CSV outputs.


In [ ]:
def run_comparison() -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    CSV_DIR.mkdir(parents=True, exist_ok=True)
    df = generate_rayleigh_data()
    X = df[INPUT_COLUMNS].to_numpy(float)
    Y = df["u_over_U"].to_numpy(float)
    all_idx = np.arange(X.shape[0])
    search_idx = all_idx
    metric_idx = all_idx
    gpr_idx = all_idx

    midl_result = run_midl_search(X, Y, search_idx)
    sidl_result = run_sidl_search(X, Y, search_idx)
    coordinates = {
        "MI-DL": {"exponents": midl_result["exponents"]},
        "SI-DL": {"exponents": sidl_result["exponents"]},
        "Known Rayleigh": {"exponents": KNOWN_RAYLEIGH},
    }
    for values in coordinates.values():
        values["log_pi"] = log_pi_from_exponents(X, values["exponents"])
        values["pi"] = np.exp(values["log_pi"])

    fits = {}
    summary_rows = []
    for method, values in coordinates.items():
        fit = gpr_fit(values["log_pi"], Y, gpr_idx)
        fits[method] = fit
        common = score_metrics(values["log_pi"][metric_idx], Y[metric_idx])
        summary_rows.append(
            {
                "method": method,
                "formula": formula_from_exponents(values["exponents"]),
                "exponents": values["exponents"],
                "n_samples": int(X.shape[0]),
                "n_search_samples": int(search_idx.size),
                "n_metric_samples": int(metric_idx.size),
                "n_gpr_samples": int(gpr_idx.size),
                "feature_space": "log_pi",
                **common,
                "gpr_mse_raw": fit["mse_raw"],
                "gpr_mse_normalized": fit["mse_normalized"],
                "gpr_rmse_normalized": fit["rmse_normalized"],
                "best_seed": midl_result["best_seed"] if method == "MI-DL" else np.nan,
            }
        )
    summary = pd.DataFrame(summary_rows)
    summary["rank_by_MI"] = summary["mutual_information"].rank(ascending=False, method="min").astype(int)
    summary["rank_by_gpr"] = summary["gpr_mse_normalized"].rank(method="min").astype(int)
    summary["rank_by_S_cov"] = summary["S_cov"].rank(ascending=False, method="min").astype(int)

    summary_csv = summary.copy()
    for j, label in enumerate(VARIABLE_LABELS):
        summary_csv[f"pi1_exp_{label}"] = summary_csv["exponents"].map(lambda arr, jj=j: arr[jj])
    summary_csv.drop(columns=["exponents"]).to_csv(CSV_DIR / "summary.csv", index=False)
    midl_result["restart_audit"].to_csv(CSV_DIR / "midl_restarts.csv", index=False)

    coord_out = df.copy()
    for method, values in coordinates.items():
        coord_out[f"{method}_log_pi1"] = values["log_pi"]
        coord_out[f"{method}_pi1"] = values["pi"]
    coord_out.to_csv(CSV_DIR / "coordinates.csv", index=False)

    exponent_rows = []
    for method, values in coordinates.items():
        for label, exponent in zip(VARIABLE_LABELS, values["exponents"]):
            exponent_rows.append(
                {
                    "method": method,
                    "variable": label,
                    "normalized_exponent": float(exponent),
                }
            )
    pd.DataFrame(exponent_rows).to_csv(CSV_DIR / "exponents.csv", index=False)

    plot_summary_table(summary)
    return {"summary": summary, "fits": fits, "coordinates": coordinates, "Y": Y}


In [ ]:
outputs = run_comparison()
outputs["summary"]


## 6. Raw-coordinate comparison

Plot the generated response directly against each learned log-coordinate. This raw scatter comparison replaces the former GPR fit figure.


In [ ]:
coordinates = outputs["coordinates"]
Y = outputs["Y"]
methods = (
    ("MI-DL", r"MI-DL $\log(\Pi_1)$"),
    ("SI-DL", r"SI-DL $\log(\Pi_1)$"),
    ("Known Rayleigh", r"Known Rayleigh $\log(\Pi_1)$"),
)
plt.rcParams.update({"font.family":"STIXGeneral","mathtext.fontset":"stix","axes.titlesize":22,"axes.labelsize":21,"xtick.labelsize":17,"ytick.labelsize":17,"legend.fontsize":15})
fig, axes = plt.subplots(1,3,figsize=(17.2,5.7),sharey=True,dpi=260)
for ax,(method,xlabel) in zip(axes,methods):
    log_pi = coordinates[method]["log_pi"]
    ax.scatter(log_pi,Y,s=42,color="#1f77b4",alpha=0.58,edgecolors="white",linewidths=0.45,label="data")
    ax.set_xlabel(xlabel)
    ax.set_title(method)
    ax.grid(True,alpha=0.22,linewidth=0.9)
    ax.legend(frameon=True,facecolor="white",edgecolor="#d1d5db",framealpha=0.94,loc="upper right")
axes[0].set_ylabel(r"$u/U$")
fig.suptitle("Rayleigh Problem",fontsize=25,y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "raw.png",bbox_inches="tight",facecolor="white")
plt.close(fig)
